# 01 — Baseline: ImageNet ResNet-50 fine-tuned on ISIC 2018 (HAM10000)

**Task.** Classify a dermoscopic image into 7 diagnostic classes (ISIC 2018 Task 3):
MEL (melanoma), NV (nevus), BCC, AKIEC, BKL, DF, VASC.

**Baseline definition** — the "default recipe" any practitioner would try first, with **no special handling of class imbalance**:

| Choice | Value |
|---|---|
| Backbone | ResNet-50, ImageNet-pretrained (`timm`), all layers fine-tuned |
| Input | images downscaled to 256 px short side, 224×224 crops (random-resized for training, center for eval) |
| Loss | plain cross-entropy (no class weighting, no resampling) |
| Optimiser | AdamW, 1-epoch linear warm-up then cosine decay |
| Model selection | best **balanced accuracy** on a lesion-grouped hold-out fold of the training set |

**Evaluation.** Official ISIC 2018 metric = **balanced multiclass accuracy** (mean per-class recall). We also report accuracy, macro-F1, macro-AUC and **melanoma sensitivity/specificity**, on:
1. `cv_val` — fold 0 of our lesion-grouped 5-fold split of the 10,015 training images (used for model selection),
2. `official_val` — the 193-image official validation set,
3. `official_test` — the 1,512-image official test set (**headline number**, comparable to the challenge leaderboard).

> Run on a GPU node (see README). Set `SMOKE_TEST=1` in the environment for a ~2-minute pipeline check on a tiny subset.

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader

HWX_DIR = Path.cwd() if (Path.cwd() / "isic").is_dir() else Path.cwd().parent
sys.path.insert(0, str(HWX_DIR))

from isic.data import CLASSES, CLASS_NAMES, ISICDataset, add_cv_folds, baseline_transforms, class_counts, load_split
from isic.engine import get_device, predict, save_json, save_predictions, seed_everything, train_one_epoch
from isic.metrics import compute_metrics, format_metrics

print("torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

torch 2.14.1+cu130 | timm 1.0.30
GPU: NVIDIA A100-SXM4-40GB


## 1. Configuration

In [2]:
CFG = dict(
    model_name="resnet50.tv2_in1k",   # torchvision ImageNet-1k weights (V2 recipe) via timm
    img_size=224,
    resize=256,
    epochs=30,
    batch_size=64,
    lr=2e-4,
    weight_decay=1e-2,
    warmup_epochs=1,
    n_folds=5,         # lesion-grouped CV split of the training set ...
    val_fold=0,        # ... fold 0 is held out for model selection
    cv_seed=0,
    seed=0,
    num_workers=min(16, os.cpu_count() or 1),
    amp=True,          # bf16 autocast
)
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
if SMOKE_TEST:
    CFG.update(epochs=2, num_workers=4)

RUN_NAME = "baseline_resnet50" + ("_smoke" if SMOKE_TEST else "")
OUT_DIR = HWX_DIR / "results" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(CFG["seed"])
device = get_device()
CFG

{'model_name': 'resnet50.tv2_in1k',
 'img_size': 224,
 'resize': 256,
 'epochs': 30,
 'batch_size': 64,
 'lr': 0.0002,
 'weight_decay': 0.01,
 'warmup_epochs': 1,
 'n_folds': 5,
 'val_fold': 0,
 'cv_seed': 0,
 'seed': 0,
 'num_workers': 16,
 'amp': True}

## 2. Data

* **Training pool**: 10,015 HAM10000 images. Several images can show the *same lesion*, so we split by `lesion_id`
  (`StratifiedGroupKFold`) — otherwise near-duplicates leak between train and validation and inflate scores.
* **Official validation** (193) and **official test** (1,512) sets are never used for training or model selection.

In [3]:
train_all = add_cv_folds(load_split("train"), n_folds=CFG["n_folds"], seed=CFG["cv_seed"])
train_df = train_all[train_all.fold != CFG["val_fold"]].reset_index(drop=True)
val_df = train_all[train_all.fold == CFG["val_fold"]].reset_index(drop=True)
off_val_df = load_split("val")
test_df = load_split("test")

if SMOKE_TEST:  # tiny class-balanced subsets
    train_df = train_df.groupby("label").head(40).reset_index(drop=True)
    val_df, off_val_df, test_df = (d.groupby("label").head(8).reset_index(drop=True) for d in (val_df, off_val_df, test_df))

# no lesion may appear on both sides of the internal split
assert not set(train_df.lesion_id) & set(val_df.lesion_id)

splits = {"train": train_df, "cv_val": val_df, "official_val": off_val_df, "official_test": test_df}
dist = pd.DataFrame({name: class_counts(df.label) for name, df in splits.items()}, index=CLASSES)
dist.loc["total"] = dist.sum()
dist["description"] = [CLASS_NAMES[c] for c in CLASSES] + [""]
dist

,train,cv_val,official_val,official_test,description
MEL,890,223,21,171,Melanoma
NV,5364,1341,123,909,Melanocytic nevus
BCC,411,103,15,93,Basal cell carcinoma
AKIEC,261,66,8,43,Actinic keratosis / Bowen's
BKL,880,219,22,217,Benign keratosis
DF,92,23,1,44,Dermatofibroma
VASC,113,29,3,35,Vascular lesion
total,8011,2004,193,1512,


In [4]:
# Class proportions: ~2/3 of the data is NV, so "always predict NV" already gets ~67% accuracy
# but only 1/7 = 14.3% balanced accuracy. This is why balanced accuracy is the official metric.
(dist.drop(index="total")[["train", "official_test"]] / dist.loc["total", ["train", "official_test"]]).round(3)

,train,official_test
MEL,0.111,0.113
NV,0.670,0.601
BCC,0.051,0.062
AKIEC,0.033,0.028
BKL,0.110,0.144
DF,0.011,0.029
VASC,0.014,0.023


In [5]:
from PIL import Image
sizes = pd.Series([Image.open(p).size for p in pd.concat([train_df.path.sample(200, random_state=0), test_df.path.head(200)])]).value_counts()
print("image sizes (W, H) in a sample:\n", sizes)

image sizes (W, H) in a sample:
 (600, 450)    400
Name: count, dtype: int64


### Datasets and loaders
Training augmentation: random-resized crop (scale 0.4–1), horizontal + vertical flips (dermoscopy has no canonical orientation),
mild color jitter, ImageNet normalisation. Evaluation: resize short side to 256 → center crop 224.

In [6]:
train_tf, eval_tf = baseline_transforms(CFG["img_size"], CFG["resize"])
loader_kw = dict(num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=CFG["num_workers"] > 0)

train_loader = DataLoader(ISICDataset(train_df, train_tf), batch_size=CFG["batch_size"], shuffle=True, drop_last=True, **loader_kw)
eval_loaders = {name: DataLoader(ISICDataset(df, eval_tf), batch_size=128, shuffle=False, **loader_kw)
                for name, df in splits.items() if name != "train"}

x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), x.dtype, "| labels:", y[:16].tolist())
print("pixel stats after normalisation: mean %.3f std %.3f" % (x.mean(), x.std()))

batch: (64, 3, 224, 224) torch.float32 | labels: [4, 1, 0, 2, 1, 1, 1, 1, 2, 0, 0, 1, 2, 1, 1, 1]
pixel stats after normalisation: mean 0.674 std 0.858


## 3. Model

In [7]:
model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=len(CLASSES))
model = model.to(device).to(memory_format=torch.channels_last)

n_params = sum(p.numel() for p in model.parameters())
print(f"{CFG['model_name']}: {n_params / 1e6:.2f}M parameters (all trainable)")
print("classifier head:", model.get_classifier())

resnet50.tv2_in1k: 23.52M parameters (all trainable)
classifier head: Linear(in_features=2048, out_features=7, bias=True)


## 4. Loss, optimiser, schedule
Plain (unweighted) cross-entropy — deliberately **no** imbalance handling, so later experiments
(class-weighted loss, focal loss, oversampling) can be compared against it.

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

steps_per_epoch = len(train_loader)
total_steps = CFG["epochs"] * steps_per_epoch
warmup_steps = CFG["warmup_epochs"] * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)  # stepped every iteration
print(f"{steps_per_epoch} steps/epoch, {total_steps} total steps")

125 steps/epoch, 3750 total steps


## 5. Training (model selection on `cv_val` balanced accuracy)

In [9]:
history = []
best_bacc, best_epoch = -1.0, -1
ckpt_path = OUT_DIR / "best.pt"
t0 = time.time()

for epoch in range(1, CFG["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device,
                                 scheduler=scheduler, amp=CFG["amp"], desc=f"epoch {epoch}")
    probs, labels = predict(model, eval_loaders["cv_val"], device, amp=CFG["amp"], desc="cv_val")
    m = compute_metrics(labels, probs)
    history.append({"epoch": epoch, "train_loss": train_loss, "lr": optimizer.param_groups[0]["lr"],
                    "val_balanced_acc": m["balanced_acc"], "val_accuracy": m["accuracy"],
                    "val_macro_auc": m["macro_auc"], "val_mel_sens": m["mel_sensitivity"]})
    if m["balanced_acc"] > best_bacc:
        best_bacc, best_epoch = m["balanced_acc"], epoch
        torch.save(model.state_dict(), ckpt_path)
    print(f"[{time.time() - t0:6.0f}s] epoch {epoch:2d} | loss {train_loss:.4f} | "
          f"val BalAcc {m['balanced_acc']:.4f} Acc {m['accuracy']:.4f} AUC {m['macro_auc']:.4f} "
          f"MEL sens {m['mel_sensitivity']:.3f} | best {best_bacc:.4f} @ {best_epoch}")

history = pd.DataFrame(history)
history.to_csv(OUT_DIR / "history.csv", index=False)
print(f"training time: {(time.time() - t0) / 60:.1f} min")

[    13s] epoch  1 | loss 1.1215 | val BalAcc 0.3787 Acc 0.7420 AUC 0.9106 MEL sens 0.731 | best 0.3787 @ 1


[    24s] epoch  2 | loss 0.5676 | val BalAcc 0.5286 Acc 0.7939 AUC 0.9471 MEL sens 0.229 | best 0.5286 @ 2


[    35s] epoch  3 | loss 0.4307 | val BalAcc 0.5755 Acc 0.8089 AUC 0.9499 MEL sens 0.224 | best 0.5755 @ 3


[    46s] epoch  4 | loss 0.3514 | val BalAcc 0.6145 Acc 0.8059 AUC 0.9578 MEL sens 0.457 | best 0.6145 @ 4


[    57s] epoch  5 | loss 0.3107 | val BalAcc 0.6801 Acc 0.8343 AUC 0.9604 MEL sens 0.327 | best 0.6801 @ 5


...

[   286s] epoch 26 | loss 0.0188 | val BalAcc 0.7149 Acc 0.8543 AUC 0.9675 MEL sens 0.484 | best 0.7465 @ 19


[   297s] epoch 27 | loss 0.0149 | val BalAcc 0.7206 Acc 0.8558 AUC 0.9660 MEL sens 0.457 | best 0.7465 @ 19


[   307s] epoch 28 | loss 0.0186 | val BalAcc 0.7141 Acc 0.8548 AUC 0.9671 MEL sens 0.466 | best 0.7465 @ 19


[   318s] epoch 29 | loss 0.0181 | val BalAcc 0.7139 Acc 0.8548 AUC 0.9668 MEL sens 0.484 | best 0.7465 @ 19


[   329s] epoch 30 | loss 0.0151 | val BalAcc 0.7190 Acc 0.8563 AUC 0.9664 MEL sens 0.498 | best 0.7465 @ 19
training time: 5.5 min


## 6. Evaluation of the best checkpoint

In [10]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))

results = {"config": CFG, "best_epoch": best_epoch, "n_params": n_params, "metrics": {}}
for name, loader in eval_loaders.items():
    probs, labels = predict(model, loader, device, amp=CFG["amp"], desc=name)
    results["metrics"][name] = compute_metrics(labels, probs)
    save_predictions(OUT_DIR / f"pred_{name}.csv", splits[name].image, probs, labels)
    print(format_metrics(results["metrics"][name], name), "\n")

save_json(results, OUT_DIR / "metrics.json")
print("saved to", OUT_DIR)

cv_val         BalAcc 0.7465 | Acc 0.8653 | F1 0.7558 | AUC 0.9683 | MEL sens 0.5426 spec 0.9691
               recall  MEL:0.54  NV:0.96  BCC:0.77  AKIEC:0.64  BKL:0.74  DF:0.65  VASC:0.93 



official_val   BalAcc 0.6840 | Acc 0.8860 | F1 0.6957 | AUC 0.9795 | MEL sens 0.7143 spec 0.9826
               recall  MEL:0.71  NV:0.95  BCC:0.93  AKIEC:0.75  BKL:0.77  DF:0.00  VASC:0.67 



official_test  BalAcc 0.7692 | Acc 0.8604 | F1 0.7832 | AUC 0.9719 | MEL sens 0.5848 spec 0.9732
               recall  MEL:0.58  NV:0.96  BCC:0.85  AKIEC:0.74  BKL:0.74  DF:0.68  VASC:0.83 

saved to /pscratch/sd/d/dhaura/repos/636_hw1_exploration/csce-636-s26-projects/hwx/results/baseline_resnet50


In [11]:
summary = pd.DataFrame({name: {k: m[k] for k in ("balanced_acc", "accuracy", "macro_f1", "macro_auc", "mel_sensitivity", "mel_specificity")}
                        for name, m in results["metrics"].items()}).T
summary.round(4)

,balanced_acc,accuracy,macro_f1,macro_auc,mel_sensitivity,mel_specificity
cv_val,0.7465,0.8653,0.7558,0.9683,0.5426,0.9691
official_val,0.6840,0.8860,0.6957,0.9795,0.7143,0.9826
official_test,0.7692,0.8604,0.7832,0.9719,0.5848,0.9732


In [12]:
# Confusion matrix on the official test set (rows = true class, columns = predicted)
cm = pd.DataFrame(results["metrics"]["official_test"]["confusion_matrix"], index=CLASSES, columns=CLASSES)
cm

,MEL,NV,BCC,AKIEC,BKL,DF,VASC
MEL,100,49,3,2,14,1,2
NV,14,871,7,2,11,4,0
BCC,3,3,79,3,3,1,1
AKIEC,1,3,3,32,2,2,0
BKL,18,26,6,1,160,4,2
DF,0,12,0,1,1,30,0
VASC,0,4,2,0,0,0,29


## 7. Context: published numbers on the same task (balanced accuracy)

| Reference | Setting | Balanced acc. |
|---|---|---|
| Gessert et al. 2018 — DenseNet-121, single model | 5-fold CV, 36-crop eval | 0.795 |
| Gessert et al. 2018 — DenseNet-121 *without* loss weighting | 5-fold CV | 0.757 |
| Gessert et al. 2018 — best single model (SENet-154) | 5-fold CV | 0.817 |
| Gessert et al. 2018 — 54-model ensemble (2nd place) | **official test** | 0.856 |
| ISIC 2018 winner (MetaOptima, used external data) | **official test** | 0.885 |
| **This baseline (ResNet-50, plain CE)** | official test | see `summary` above |

Our re-implementation of the Gessert et al. single-model recipe lives in `literature_baseline_gessert2018.py`.

**Next experiments** (later notebooks): imbalance handling (weighted CE / focal / oversampling), stronger and
domain-specific backbones (ViT / DINOv2 / BiomedCLIP), linear probe vs full fine-tuning, multi-crop evaluation.